In [9]:
import pdfplumber
import pandas as pd
import numpy as np

In [ ]:
all_sub_tables = []
with pdfplumber.open('/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Papers/Schneider and Medina (2018) - Shadow Economies Around the World - What did we Learn Over the Last 20 Years.pdf') as pdf:
    for p in pdf.pages[60:]:
        for t in p.extract_tables():
            # print(t)
            all_sub_tables.append(t)
# print(all_sub_tables)

In [11]:
from collections import Counter


class ColumnRenamer:

    def __init__(self, separator=None):
        self.counter = Counter()
        self.sep = separator

    def __call__(self, x):
        index = self.counter[x]  # Counter returns 0 for missing elements
        self.counter[x] = index + 1  # Uses something like `setdefault`
        return f'{x}{self.sep if self.sep and index else ""}{index if index else ""}'


In [18]:
df = pd.DataFrame(columns = all_sub_tables[0][0][1:])
df2 = pd.DataFrame(columns = all_sub_tables[8][0][1:])

for sub_table in all_sub_tables[0:8]:
    for row in sub_table[1:]:
        df.loc[len(df)] = row[1:]
for sub_table in all_sub_tables[8:]:
    for row in sub_table[1:]:
        df2.loc[len(df2)] = row[1:]

final_dataset = pd.concat([df, df2], axis = 1)

final_dataset = final_dataset.replace({'\n': ' '}, regex=True)
final_dataset = final_dataset.rename(columns = {'Av.\nover\nyears': 'Av. over years'})
print(final_dataset.columns)
final_dataset = final_dataset.rename(columns = ColumnRenamer(separator = '_'))
final_dataset = final_dataset.drop('Country_1', axis = 1)
print(final_dataset.columns)

final_dataset.to_csv('size_of_the_shadow_economy.csv', index = False)

Index(['Country', '1991', '1992', '1993', '1994', '1995', '1996', '1997',
       '1998', '1999', '2000', '2001', '2002', '2003', 'Country', '2004',
       '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013',
       '2014', '2015', 'Av. over years'],
      dtype='str')
Index(['Country', '1991', '1992', '1993', '1994', '1995', '1996', '1997',
       '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006',
       '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015',
       'Av. over years'],
      dtype='str')


#### Dealing with Naming Issues

In [ ]:
names_conversion_dictionary = {}
with open("country_name_changes_se_size_dataset.txt") as inp:
    lines_from_txt = list(inp)

for item in lines_from_txt:
    names_conversion_dictionary[item.split(' -> ')[0]] = item.split(' -> ')[1].removesuffix('\n')
# print(names_conversion_dictionary)

final_dataset_with_country_name_changes = final_dataset.copy()
final_dataset_with_country_name_changes['Country'] = final_dataset['Country'].apply(lambda x: names_conversion_dictionary[x] if x in names_conversion_dictionary != None else x)
final_dataset_with_country_name_changes.to_csv('size_of_the_shadow_economy.csv', index = False)